# Rubric conditioning: can a grader score a criterion it has never seen?

Runs `backend/scripts/rubric_conditioning.py` on a Colab T4. The design is pre-declared in `rubric_conditioning_results.md` and committed; nothing here may change in response to the numbers.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells in order. **Step 6 scores the locked 8-essay holdout: run it once.** The script refuses a second run into the same output folder.

In [ ]:
# 1. Confirm the GPU
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Point at the uploaded folder and check it is all there
ROOT = "/content/drive/MyDrive/rubric_conditioning"
!ls -la "{ROOT}/backend/scripts" "{ROOT}/data"

In [ ]:
# 4. Install dependencies (torch is Colab's own CUDA build; this normally installs nothing)
!pip install -q -r "{ROOT}/backend/requirements-colab.txt"

In [ ]:
# 5a. Pre-flight only: checks the holdout is the locked file, that its criterion
#     never appears in training, and that no essay is in the training text.
#     Trains nothing. Expect: PRE-FLIGHT PASSED
!python "{ROOT}/backend/scripts/rubric_conditioning.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output" --dry-run

In [ ]:
# 5b. Smoke test: one learning rate, one epoch, a slice of the data. Exercises every
#     code path cheaply. Its numbers mean nothing — it writes to a separate folder.
!python "{ROOT}/backend/scripts/rubric_conditioning.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output_smoke" --smoke

In [ ]:
# 6. THE RUN — run once. Trains both variants and scores the locked holdout.
#    Expect roughly 15–20 minutes: two models x two learning rates x 4 epochs.
!python "{ROOT}/backend/scripts/rubric_conditioning.py" --data-dir "{ROOT}/data" --out-dir "{ROOT}/output"

In [ ]:
# 7. Show the headline numbers, then flush Drive so nothing is lost
import json
r = json.load(open(f"{ROOT}/output/rubric_conditioning.json"))
for name, m in r["zero_shot"].items():
    print(f"{name:20} MAE {m['mae']:.2f}  QWK {m['qwk']:.2f}  within-1 {m['within_one']:.0%}")
print()
for metric, c in r["a_vs_b"].items():
    mark = "distinguishable" if c["distinguishable"] else "NOT distinguishable"
    print(f"A-B {metric:11} {c['a_minus_b']:+.3f} [{c['ci'][0]:+.3f}, {c['ci'][1]:+.3f}] — {mark}")

drive.flush_and_unmount()
print("\nDone. Download output/rubric_conditioning.json and output/rubric_conditioning_log.txt.")